# Study of prediction & reconstruction performance 

## *Experiences description*

### Experiment 3 - Autoencoders Trained on Increasing Climate Diversity

In this third experiment, we want to study the distribution shift of our data across climates inside an autoencoder (AE). So we focus on the latent representation of an AE trained on some climates.

We retrieve the patchs used in the second experiment. We randomly split these samples in train/val/test datasets for each climate.
We then train the AE on one of these configurations (using the train and validation sets) :
- historical climate
- historical and ssp245 climates
- historical, ssp245 and ssp370 climates
- all climates

Then we evaluate the reconstruction quality of the AE on all climates (using the test sets)

We then project the latent representations of the test sets into a common PCA space, and we plot some visualization of this PCA space. 

And finally we perform the same analyzes between distributions then in the second experiment : 
- compute multivariate shift metrics in that latent PCA space;
- analyze the moments of the latent principal components;
- analyze extreme latent scores;
- analyze seasonal shifts in latent PCA space.

### Experiment 4 - Invariant Autoencoder with Latent Alignment

In this fourth experiment, we dive a step closer to the real CERA architecture by adding an explicit climate invariance term to the autoencoder loss.

The idea is to test wether adding an alignment loss between climates will effectivelly bring different climate distributions closer compared to the raw data and to the simple AE architecture. Here we only train the AE on the historical climate and on SSP245 to reproduce the CERA architecture (one "cold" and one "warm" climate).

Training set:
- **historical + ssp245**

Loss:
- reconstruction loss on all samples;
- alignment loss between latent samples from **historical** and **ssp245**.

Loss equation : 
$$
L = L_{\text{rec}} + \lambda_{\text{Align}} \cdot \text{Align}(Z^{\text{hist}}_{\text{align}}, Z^{\text{ssp245}}_{\text{align}})
$$

Alignment method : 

Here we consider two different method to align the historical and SSP245 climates ;
- We consider a sliced Wasserstein alignment loss 
- And a adversarial classifier.

Note that in CERA, the method used is Earth Mover's Distance (EMD), but EMD can be expensive in high dimension, it is why we use a sliced Wasserstein alignment loss, which is a practical EMD-style approximation.

Interpretation:
- decreasing the alignment term should reduce latent distribution shift between historical and ssp245, and potentially between historical and other warmer climates;
- this must be balanced against reconstruction quality;
- PCA visualizations help determine whether the latent clouds become more mixed and allow us to calculate the same metrics as before onto our "normalized" PCA space.

### Experiment 5 - CERA-like architecture

In this fifth experiment, we add a predictor to the architecture considered in the fourth experiment. We thus now consider : AE (constructed either with cnn2D or MLPs) (and with either sliced wasserstein distance alignment or adversarial classifier alignment) and a predictor using only the aligned part of the historical climate latent representations. The predictor needs to predict a given variable field over the whole grid of the samples. This architecture will be called a CERA-like architecture.

The same test/train/val split than before is used.

This CERA-like setup extends the invariant AE by adding a precipitation predictor:
- AE input: multivariate samples from historical + ssp245 climates.
- AE losses: reconstruction + latent alignment on the first 48 latent dimensions.
- Predictor: MLP on aligned latent dimensions (historical only) to predict a given variable over all 70 patch points.

Global loss used for AE update:
$$
L_{AE} = L_{rec} + \lambda_{align} L_{align} + \lambda_{pred} L_{pred}
$$

And we perform the predictor update at the same time, to be consistent with the end to end training used in the original CERA architecture.

## Part 0 - Global Configuration

**Library import**

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import to_rgba
from IPython.display import display
import seaborn as sns
import json
import gc
from pathlib import Path
from sklearn.metrics import r2_score, mean_squared_error
from pathlib import Path
import pickle
from matplotlib.lines import Line2D
from scipy.stats import gaussian_kde
from matplotlib.colors import Normalize
from matplotlib.colors import LinearSegmentedColormap, TwoSlopeNorm
from scipy.ndimage import gaussian_filter1d

**Data Loading**

In [ ]:
num_sample = 1000000
chosen_autoencoder_type = "CNN" # choose between "MLP" and "CNN"
inv_alignment_method = "swd" # choose between "swd", "swdn" and "adversarial"
variable = "pr" # variable to predict
val_fraction = 0.05
test_fraction = 0.15

In [ ]:
precomputed_dir = Path(f"/glade/derecho/scratch/tsalin/CMIP/derived/multivariate_samples_optimized_NGS_v1000")
if not precomputed_dir.exists():
    raise FileNotFoundError(f"Precomputed data directory not found: {precomputed_dir}")

with open(precomputed_dir / "run_config.json", "r", encoding="utf-8") as f:
    run_cfg = json.load(f)

climate_order = list(run_cfg["climate_order"])
climate_colors = dict(run_cfg["climate_colors"])
selected_variables_full = list(run_cfg["selected_variables"])
max_abs_lat = float(run_cfg["max_abs_lat"])
patch_size_km = float(run_cfg["patch_size_km"])
time_stride = int(run_cfg["time_stride"])
max_samples_per_climate = int(run_cfg["max_samples_per_climate"])
random_seed = int(run_cfg["random_seed"])
n_lat = int(run_cfg["n_lat"])
n_lon = int(run_cfg["n_lon"])
grid_points_per_patch = int(run_cfg["grid_points_per_patch"])
n_patches = int(run_cfg["n_patches"])

We import the evaluation DataFrames

In [ ]:
evaluation_root = Path("/glade/work/tsalin/CMIP/model_evaluation")
if not evaluation_root.exists():
    raise FileNotFoundError(f"Model evaluation directory not found: {evaluation_root}")


def _load_quality_payload(file_path):
    """Load a quality payload dict from a pkl file."""
    file_path = Path(file_path)
    if not file_path.exists():
        raise FileNotFoundError(f"Model evaluation file not found: {file_path}")
    with open(file_path, "rb") as fh:
        content = pickle.load(fh)
    if not isinstance(content, dict):
        raise TypeError(f"Expected a dict payload in {file_path.name}, got {type(content)}")
    return content


def _load_history_df(file_path):
    """Load a history DataFrame from a pkl file."""
    file_path = Path(file_path)
    if not file_path.exists():
        raise FileNotFoundError(f"Model evaluation file not found: {file_path}")
    with open(file_path, "rb") as fh:
        content = pickle.load(fh)
    if not isinstance(content, pd.DataFrame):
        raise TypeError(f"Expected a DataFrame in {file_path.name}, got {type(content)}")
    return content.copy()


def _cera_quality_path(lambda_align, lambda_pred):
    prefix = (
        f"cera_seasonal_ns{num_sample}_{chosen_autoencoder_type}_{inv_alignment_method}_"
        f"{variable}_{val_fraction}_{test_fraction}_{lambda_align}_{lambda_pred}_"
    )
    return evaluation_root / "CERA_seasonal" / f"{prefix}quality_df.pkl"


def _cera_history_path(lambda_align, lambda_pred):
    prefix = (
        f"cera_seasonal_ns{num_sample}_{chosen_autoencoder_type}_{inv_alignment_method}_"
        f"{variable}_{val_fraction}_{test_fraction}_{lambda_align}_{lambda_pred}_"
    )
    return evaluation_root / "CERA_seasonal" / f"{prefix}history_df.pkl"


# (lambda_align, lambda_pred) pairs to analyze — one CERA run per pair.
# The "quality" payloads are loaded one at a time, setup by setup, in Part 1
# (to never keep more than one complete setup in RAM). The "history" files
# are only loaded in Part 4, where they are used.
lambda_pairs = [
    (0.1, 0.1),
    (0.25, 0.1),
    (0.45, 0.1),
    (0.65, 0.1),
    (0.85, 0.1),
    (0.25, 0.25),
    (0.45, 0.25),
    (0.65, 0.25),
    (0.1, 0.45),
    (0.25, 0.45),
    (0.45, 0.45),
    (0.1, 0.65),
    (0.25, 0.65),
    (0.1, 0.85),
]

## Part 1 - Evaluation metrics computation

In [ ]:
def _safe_r2(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    mask = np.isfinite(y_true) & np.isfinite(y_pred)
    y_true, y_pred = y_true[mask], y_pred[mask]
    if y_true.size < 2:
        return np.nan
    return float(r2_score(y_true, y_pred))


def _safe_rmse(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    mask = np.isfinite(y_true) & np.isfinite(y_pred)
    y_true, y_pred = y_true[mask], y_pred[mask]
    if y_true.size == 0:
        return np.nan
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))


def _parse_variable_slices(value_names):
    """Return list of (var_name, slice) in column order, one entry per unique variable."""
    slices = []
    seen = {}
    for i, name in enumerate(value_names):
        var = str(name).split("@")[0]
        if var not in seen:
            seen[var] = i
            slices.append((var, i))
    result = []
    for k, (var, start) in enumerate(slices):
        end = slices[k + 1][1] if k + 1 < len(slices) else len(value_names)
        result.append((var, slice(start, end)))
    return result


def _compute_scores_for_component(meta_df, truth_arr, pred_arr, value_names,
                                   compute_sample_r2=True, keep_value_names=True):
    """
    Vectorized score computation for one component (reconstruction or prediction).
    Returns a DataFrame with metadata + scores + truth/pred as numpy rows.
    compute_sample_r2=False skips the per-sample R² column ("r2"), which is not
    needed when only summarize_setup_scores() will consume the output.
    """
    var_slices = _parse_variable_slices(value_names)
    N = len(meta_df)
    meta_reset = meta_df.reset_index(drop=True)

    scenario_indices = {
        scenario: grp.index.to_numpy()
        for scenario, grp in meta_reset.groupby("scenario", sort=False)
    }

    if compute_sample_r2:
        # Per-sample R²: use per-climate column mean as baseline (avoids division by zero)
        sample_r2_by_var = {}
        for var, sl in var_slices:
            t = truth_arr[:, sl].astype(np.float64)
            p = pred_arr[:, sl].astype(np.float64)
            ss_res = np.nansum((t - p) ** 2, axis=1)
            t_mean_by_sample = np.empty_like(t)
            for idx in scenario_indices.values():
                t_mean_by_sample[idx] = np.nanmean(t[idx], axis=0, keepdims=True)
            ss_tot = np.nansum((t - t_mean_by_sample) ** 2, axis=1)
            with np.errstate(invalid="ignore", divide="ignore"):
                r2 = np.where(ss_tot > 0, 1.0 - ss_res / ss_tot, np.nan)
            sample_r2_by_var[var] = r2  # [N]

        var_names = [v for v, _ in var_slices]
        sample_r2_matrix = np.column_stack([sample_r2_by_var[var] for var in var_names])
        sample_r2_list = [
            dict(zip(var_names, row.tolist()))
            for row in sample_r2_matrix
        ]

    # Global R²/RMSE per scenario (numpy slicing, no row iteration)
    global_scores = {}
    for scenario, idx in scenario_indices.items():
        r2g, rmseg = {}, {}
        for var, sl in var_slices:
            t_all = truth_arr[idx, sl].ravel().astype(np.float64)
            p_all = pred_arr[idx, sl].ravel().astype(np.float64)
            r2g[var]   = _safe_r2(t_all, p_all)
            rmseg[var] = _safe_rmse(t_all, p_all)
        global_scores[scenario] = {"r2_global": r2g, "rmse_global": rmseg}

    out = meta_reset.copy()
    if compute_sample_r2:
        out["r2"] = sample_r2_list
    out["r2_global"]   = out["scenario"].map(lambda s: global_scores[s]["r2_global"])
    out["rmse_global"] = out["scenario"].map(lambda s: global_scores[s]["rmse_global"])
    # Store numpy rows (views into the original array — no data duplication)
    out["truth_values"] = list(truth_arr)
    out["pred_values"]  = list(pred_arr)
    if keep_value_names:
        out["value_names"] = [list(value_names)] * N
    return out


def add_sample_and_global_scores(payload, compute_sample_r2=True, keep_value_names=True):
    """
    Takes a quality payload dict (new numpy format) and returns a complete DataFrame
    with per-sample and global R²/RMSE scores.
    """
    frames = []

    if "meta_reconstruction" in payload:
        frames.append(_compute_scores_for_component(
            payload["meta_reconstruction"],
            payload["truth_reconstruction"],
            payload["pred_reconstruction"],
            payload["reconstruction_value_names"],
            compute_sample_r2=compute_sample_r2,
            keep_value_names=keep_value_names,
        ))

    if "meta_prediction" in payload:
        frames.append(_compute_scores_for_component(
            payload["meta_prediction"],
            payload["truth_prediction"],
            payload["pred_prediction"],
            payload["prediction_value_names"],
            compute_sample_r2=compute_sample_r2,
            keep_value_names=keep_value_names,
        ))

    if not frames:
        raise ValueError("Payload has neither 'meta_reconstruction' nor 'meta_prediction'.")

    return pd.concat(frames, ignore_index=True)




EXTREME_PRECIP_THRESHOLD = 10.0  # mm/day — Very heavy rain


def summarize_setup_scores(complete_df, lambda_align, lambda_pred, variable,
                            extreme_threshold=EXTREME_PRECIP_THRESHOLD):
    """
    Collapse a full per-sample quality DataFrame (one setup) into a tiny summary:
    one row per (component, scenario), keeping only r2_global/rmse_global (already
    constant within such a group) plus, for component == "prediction" and
    variable == "pr", the RMSE restricted to extreme precipitation samples
    (truth >= extreme_threshold). This is the only thing kept in memory once the
    full per-sample DataFrame for this setup is discarded.
    """
    required_columns = {"component", "scenario", "r2_global", "rmse_global"}
    missing_columns = required_columns - set(complete_df.columns)
    if missing_columns:
        raise KeyError(f"Missing columns {sorted(missing_columns)} in setup dataframe.")

    summary = (
        complete_df
        .groupby(["component", "scenario"], as_index=False, sort=False)
        .agg(r2_global=("r2_global", "first"), rmse_global=("rmse_global", "first"))
    )
    summary.insert(0, "lambda_pred", lambda_pred)
    summary.insert(0, "lambda_align", lambda_align)

    summary["rmse_extreme"] = np.nan
    if variable == "pr":
        pred_df = complete_df[complete_df["component"] == "prediction"]
        for scenario, climate_df in pred_df.groupby("scenario", sort=False):
            truth_arr = np.concatenate(
                [np.asarray(v, dtype=float).ravel() for v in climate_df["truth_values"]]
            )
            pred_arr = np.concatenate(
                [np.asarray(v, dtype=float).ravel() for v in climate_df["pred_values"]]
            )
            finite_mask = np.isfinite(truth_arr) & np.isfinite(pred_arr)
            truth_arr = truth_arr[finite_mask]
            pred_arr = pred_arr[finite_mask]
            if truth_arr.size == 0:
                continue

            ext_mask = truth_arr >= extreme_threshold
            if not np.any(ext_mask):
                continue

            rmse_ext = float(np.sqrt(np.mean((truth_arr[ext_mask] - pred_arr[ext_mask]) ** 2)))
            row_mask = (summary["component"] == "prediction") & (summary["scenario"] == scenario)
            summary.loc[row_mask, "rmse_extreme"] = rmse_ext

    return summary


In [ ]:
_summary_rows = []

for lambda_align, lambda_pred in lambda_pairs:
    quality_path = _cera_quality_path(lambda_align, lambda_pred)

    payload = _load_quality_payload(quality_path)
    complete_df = add_sample_and_global_scores(payload, compute_sample_r2=False, keep_value_names=False)
    del payload

    _summary_rows.append(summarize_setup_scores(complete_df, lambda_align, lambda_pred, variable))

    del complete_df
    gc.collect()

complete_summary_df = pd.concat(_summary_rows, ignore_index=True)
del _summary_rows
gc.collect()

complete_summary_df


## Part 2 - Plotting quality results - reconstruction

In [ ]:
setup_order = [f"{lambda_align:g}_{lambda_pred:g}" for lambda_align, lambda_pred in lambda_pairs]

climate_offsets = {
    "historical": (0, 0),
    "ssp245": (0, 1),
    "ssp370": (1, 0),
    "ssp585": (1, 1),
}

reconstruction_df = complete_summary_df[complete_summary_df["component"] == "reconstruction"].copy()
if reconstruction_df.empty:
    raise ValueError("No reconstruction rows found in complete_summary_df.")

reconstruction_df["setup"] = [
    f"{lambda_align:g}_{lambda_pred:g}"
    for lambda_align, lambda_pred in zip(reconstruction_df["lambda_align"], reconstruction_df["lambda_pred"])
]

r2_heatmap_rows = []
variable_order = []
seen_variables = set()

for _, row in reconstruction_df.iterrows():
    climate = str(row["scenario"])
    if climate not in climate_offsets:
        continue

    r2_global = row["r2_global"]
    if not isinstance(r2_global, dict):
        raise TypeError(f"Expected r2_global to be a dictionary in setup {row['setup']} / {climate}.")

    for reconstructed_variable, r2_value in r2_global.items():
        reconstructed_variable = str(reconstructed_variable)
        if reconstructed_variable == variable:
            continue

        if reconstructed_variable not in seen_variables:
            seen_variables.add(reconstructed_variable)
            variable_order.append(reconstructed_variable)

        r2_heatmap_rows.append({
            "setup": row["setup"],
            "variable": reconstructed_variable,
            "climate": climate,
            "r2_global": r2_value,
        })

if not variable_order:
    raise ValueError("No reconstructed variables found for the heatmap.")

heatmap_matrix = np.full((2 * len(variable_order), 2 * len(setup_order)), np.nan, dtype=float)
setup_index = {setup_name: idx for idx, setup_name in enumerate(setup_order)}
variable_index = {reconstructed_variable: idx for idx, reconstructed_variable in enumerate(variable_order)}

for row in r2_heatmap_rows:
    setup_idx = setup_index[row["setup"]]
    variable_idx = variable_index[row["variable"]]
    climate_row, climate_col = climate_offsets[row["climate"]]
    heatmap_matrix[2 * variable_idx + climate_row, 2 * setup_idx + climate_col] = row["r2_global"]

finite_values = heatmap_matrix[np.isfinite(heatmap_matrix)]
if finite_values.size == 0:
    raise ValueError("No finite r2_global values found for the reconstruction heatmap.")

cmap = plt.get_cmap("Blues").copy()
cmap.set_bad("#FFFFFF")

fig, ax = plt.subplots(
    figsize=(max(11, 1.45 * len(setup_order) + 3), max(7, 0.55 * len(variable_order) + 2.5)),
    constrained_layout=True,
)
im = ax.imshow(heatmap_matrix, aspect="auto", cmap=cmap, vmin=float(finite_values.min()), vmax=1.0, origin="upper")

ax.set_xticks(np.arange(len(setup_order)) * 2 + 0.5)
ax.set_xticklabels(setup_order, rotation=25, ha="right")
ax.set_yticks(np.arange(len(variable_order)) * 2 + 0.5)
ax.set_yticklabels(variable_order)

ax.set_xlabel("Setup ($\\lambda_{align}$_$\\lambda_{pred}$)")
ax.set_ylabel("Reconstructed variable")
ax.set_title(
    "Global $R^2$ for reconstruction\n"
    "Quadrants: historical (top-left), ssp245 (top-right), ssp370 (bottom-left), ssp585 (bottom-right)"
)

ax.set_xticks(np.arange(-0.5, heatmap_matrix.shape[1], 1), minor=True)
ax.set_yticks(np.arange(-0.5, heatmap_matrix.shape[0], 1), minor=True)
ax.grid(which="minor", color="white", linewidth=1.0)
ax.tick_params(which="minor", bottom=False, left=False)

for setup_boundary in np.arange(2, heatmap_matrix.shape[1], 2):
    ax.axvline(setup_boundary - 0.5, color="#111827", linewidth=1.3)
for variable_boundary in np.arange(2, heatmap_matrix.shape[0], 2):
    ax.axhline(variable_boundary - 0.5, color="#111827", linewidth=1.3)

colorbar = fig.colorbar(im, ax=ax, fraction=0.03, pad=0.02)
colorbar.set_label("Global $R^2$")

plt.show()

## Part 3 - Plotting quality results - prediction

In [ ]:
from matplotlib.colors import LinearSegmentedColormap

# ── Source: the small summary computed setup by setup (Part 1) ───────────────
prediction_df = complete_summary_df[complete_summary_df["component"] == "prediction"].copy()
if prediction_df.empty:
    raise ValueError("No prediction rows found in complete_summary_df.")

prediction_df["r2_value"] = prediction_df["r2_global"].apply(
    lambda d: d.get(variable, np.nan) if isinstance(d, dict) else d
)

scenario_order = list(dict.fromkeys(prediction_df["scenario"].tolist()))
align_values = sorted(prediction_df["lambda_align"].unique())
pred_values = sorted(prediction_df["lambda_pred"].unique(), reverse=True)  # descending -> top of the heatmap = highest pred

# ── Diverging palette (blue = good, red = bad), centered on 0 ───────────
diverging_cmap = LinearSegmentedColormap.from_list(
    "diverging_blue_red", ["#e34948", "#f0efec", "#2a78d6"], N=256
)
diverging_cmap.set_bad("#d9d9d6")  # missing cells (align/pred combination absent)

def _make_pivot(values_df, value_col):
    pivot = values_df.pivot_table(index="lambda_pred", columns="lambda_align", values=value_col, aggfunc="first")
    pivot = pivot.reindex(index=pred_values, columns=align_values)
    return pivot

def _plot_heatmap(ax, pivot, title, cmap, vmin=None, vmax=None, norm=None, fmt="{:.2f}"):
    im = ax.imshow(pivot.values, cmap=cmap, vmin=vmin, vmax=vmax, norm=norm, aspect="auto")
    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels([str(c) for c in pivot.columns], rotation=45, ha="right")
    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels([str(i) for i in pivot.index])
    ax.set_xlabel(r"$\lambda_{align}$")
    ax.set_ylabel(r"$\lambda_{pred}$")
    ax.set_title(title, fontsize=10)

    for i in range(pivot.shape[0]):
        for j in range(pivot.shape[1]):
            val = pivot.values[i, j]
            if not np.isnan(val):
                ax.text(j, i, fmt.format(val), ha="center", va="center", fontsize=8, color="black")
    return im

# ── 1) One R² heatmap per climate (scenario) ──────────────────────────────────
r2_vmin, r2_vmax = -0.5, 1.0
n_scenarios = len(scenario_order)
fig, axes = plt.subplots(1, n_scenarios, figsize=(4 * n_scenarios, 4.2), constrained_layout=True)
if n_scenarios == 1:
    axes = [axes]

im = None
for ax, scenario in zip(axes, scenario_order):
    scenario_df = prediction_df[prediction_df["scenario"] == scenario]
    pivot = _make_pivot(scenario_df, "r2_value")
    im = _plot_heatmap(ax, pivot, f"R² – {scenario}", diverging_cmap, r2_vmin, r2_vmax)

fig.colorbar(im, ax=axes, shrink=0.8, label=f"Global $R^2$ for {variable}")
fig.suptitle(f"Global $R^2$ by scenario ($\\lambda_{{align}}$ vs $\\lambda_{{pred}}$) – {variable}", fontsize=12)
plt.show()

# ── 2) One Δ R² heatmap (vs historical) per SSP scenario ─────────────────────
_ref_col = next((c for c in ("historical", "hist") if c in scenario_order), None)
if _ref_col is None:
    raise ValueError(f"Historical scenario not found. Available scenarios: {scenario_order}")

_ssp_scenarios = [s for s in scenario_order if s != _ref_col]
if not _ssp_scenarios:
    raise ValueError("No SSP scenario found in the data.")

_ref_df = prediction_df[prediction_df["scenario"] == _ref_col][["lambda_align", "lambda_pred", "r2_value"]]
_ref_df = _ref_df.rename(columns={"r2_value": "r2_ref"})

# A single r2_value per (lambda_align, lambda_pred, scenario) is guaranteed by summarize_setup_scores,
# so this merge stays bounded (no risk of combinatorial explosion).
_delta_long = prediction_df[prediction_df["scenario"].isin(_ssp_scenarios)].merge(
    _ref_df, on=["lambda_align", "lambda_pred"], how="left"
)
_delta_long["delta_r2"] = _delta_long["r2_value"] - _delta_long["r2_ref"]

delta_vmin, delta_vmax = -1.5, 0.1
n_ssp = len(_ssp_scenarios)
fig2, axes2 = plt.subplots(1, n_ssp, figsize=(4 * n_ssp, 4.2), constrained_layout=True)
if n_ssp == 1:
    axes2 = [axes2]

im2 = None
for ax, scenario in zip(axes2, _ssp_scenarios):
    scenario_df = _delta_long[_delta_long["scenario"] == scenario]
    pivot = _make_pivot(scenario_df, "delta_r2")
    im2 = _plot_heatmap(ax, pivot, f"Δ R² – {scenario}", diverging_cmap, delta_vmin, delta_vmax)

fig2.colorbar(im2, ax=axes2, shrink=0.8, label=f"Δ Global $R^2$ for {variable}")
fig2.suptitle(f"Δ Global $R^2$ vs. {_ref_col} ($\\lambda_{{align}}$ vs $\\lambda_{{pred}}$) – {variable}", fontsize=12)
plt.show()


In [ ]:
from matplotlib.colors import LinearSegmentedColormap, TwoSlopeNorm

# ── Source: the small summary computed setup by setup (Part 1) ───────────────
prediction_df = complete_summary_df[complete_summary_df["component"] == "prediction"].copy()
if prediction_df.empty:
    raise ValueError("No prediction rows found in complete_summary_df.")

prediction_df["r2_value"] = prediction_df["r2_global"].apply(
    lambda d: d.get(variable, np.nan) if isinstance(d, dict) else d
)

# ── Restriction to the half-square (0.1,0.1) → (0.85,0.1) → (0.1,0.85) ────────────
_TRI_ALIGN_MIN, _TRI_PRED_MIN = 0.1, 0.1
_TRI_HYPOTENUSE = 0.95  # = 0.85 + 0.1, the constant sum along the hypotenuse

def _filter_triangle(df):
    mask = (
        (df["lambda_align"] >= _TRI_ALIGN_MIN - 1e-9) &
        (df["lambda_pred"] >= _TRI_PRED_MIN - 1e-9) &
        (df["lambda_align"] + df["lambda_pred"] <= _TRI_HYPOTENUSE + 1e-9)
    )
    return df[mask]

prediction_df = _filter_triangle(prediction_df)
if prediction_df.empty:
    raise ValueError("No prediction rows inside the (0.1,0.1)-(0.85,0.1)-(0.1,0.85) triangle.")

scenario_order = list(dict.fromkeys(prediction_df["scenario"].tolist()))

# ── Diverging palette (red = heatmap min value, blue = heatmap max value) ──
diverging_cmap = LinearSegmentedColormap.from_list(
    "diverging_blue_red", ["#e34948", "#f0efec", "#2a78d6"], N=256
)

def _plot_continuous_heatmap(ax, fig, values_df, value_col, title, cmap, kind="minmax", fmt="{:.2f}", cbar_label=""):
    x = values_df["lambda_align"].to_numpy()
    y = values_df["lambda_pred"].to_numpy()
    z = values_df[value_col].to_numpy()

    vmin = float(np.nanmin(z))
    vmax = float(np.nanmax(z))

    if kind == "diverging_center0":
        norm = TwoSlopeNorm(vmin=min(vmin, -1e-6), vcenter=0.0, vmax=max(vmax, 1e-6))
        tcf = ax.tricontourf(x, y, z, levels=21, cmap=cmap, norm=norm)
    else:
        if vmin == vmax:
            vmin, vmax = vmin - 1e-6, vmax + 1e-6
        levels = np.linspace(vmin, vmax, 21)
        tcf = ax.tricontourf(x, y, z, levels=levels, cmap=cmap, vmin=vmin, vmax=vmax)

    ax.scatter(x, y, s=16, facecolor="white", edgecolor="black", linewidth=0.6, zorder=3)
    for xi, yi, zi in zip(x, y, z):
        if np.isfinite(zi):
            ax.annotate(fmt.format(zi), (xi, yi), textcoords="offset points", xytext=(4, 4), fontsize=7)

    ax.set_xlabel(r"$\lambda_{align}$")
    ax.set_ylabel(r"$\lambda_{pred}$")
    ax.set_title(title, fontsize=10)
    ax.set_xlim(_TRI_ALIGN_MIN, 0.85)
    ax.set_ylim(_TRI_PRED_MIN, 0.85)
    ax.set_aspect("equal", adjustable="box")

    fig.colorbar(tcf, ax=ax, shrink=0.85, label=cbar_label)
    return tcf

# ── 1) One continuous R² heatmap per climate (scenario), own scale for each ─
n_scenarios = len(scenario_order)
fig, axes = plt.subplots(1, n_scenarios, figsize=(4.6 * n_scenarios, 4.2), constrained_layout=True)
if n_scenarios == 1:
    axes = [axes]

for ax, scenario in zip(axes, scenario_order):
    scenario_df = prediction_df[prediction_df["scenario"] == scenario]
    _plot_continuous_heatmap(
        ax, fig, scenario_df, "r2_value", f"R² – {scenario}", diverging_cmap,
        kind="minmax", cbar_label=f"$R^2$ for {variable}",
    )

fig.suptitle(f"Global $R^2$ by scenario ($\\lambda_{{align}}$ vs $\\lambda_{{pred}}$, half-square) – {variable}", fontsize=12)
plt.show()

# ── 2) One continuous Δ R² heatmap (vs historical) per SSP scenario, own scale for each ─
_ref_col = next((c for c in ("historical", "hist") if c in scenario_order), None)
if _ref_col is None:
    raise ValueError(f"Historical scenario not found. Available scenarios: {scenario_order}")

_ssp_scenarios = [s for s in scenario_order if s != _ref_col]
if not _ssp_scenarios:
    raise ValueError("No SSP scenario found in the data.")

_ref_df = prediction_df[prediction_df["scenario"] == _ref_col][["lambda_align", "lambda_pred", "r2_value"]]
_ref_df = _ref_df.rename(columns={"r2_value": "r2_ref"})

# A single r2_value per (lambda_align, lambda_pred, scenario) is guaranteed by summarize_setup_scores,
# so this merge stays bounded (no risk of combinatorial explosion).
_delta_long = prediction_df[prediction_df["scenario"].isin(_ssp_scenarios)].merge(
    _ref_df, on=["lambda_align", "lambda_pred"], how="left"
)
_delta_long["delta_r2"] = _delta_long["r2_value"] - _delta_long["r2_ref"]

n_ssp = len(_ssp_scenarios)
fig2, axes2 = plt.subplots(1, n_ssp, figsize=(4.6 * n_ssp, 4.2), constrained_layout=True)
if n_ssp == 1:
    axes2 = [axes2]

for ax, scenario in zip(axes2, _ssp_scenarios):
    scenario_df = _delta_long[_delta_long["scenario"] == scenario]
    _plot_continuous_heatmap(
        ax, fig2, scenario_df, "delta_r2", f"Δ R² – {scenario}", diverging_cmap,
        kind="diverging_center0", cbar_label=f"Δ $R^2$ for {variable}",
    )

fig2.suptitle(f"Δ Global $R^2$ vs. {_ref_col} ($\\lambda_{{align}}$ vs $\\lambda_{{pred}}$, half-square) – {variable}", fontsize=12)
plt.show()


In [ ]:
from matplotlib.colors import LinearSegmentedColormap, TwoSlopeNorm

# ── Source: the small summary computed setup by setup (Part 1) ───────────────
prediction_df = complete_summary_df[complete_summary_df["component"] == "prediction"].copy()
if prediction_df.empty:
    raise ValueError("No prediction rows found in complete_summary_df.")

prediction_df["rmse_value"] = prediction_df["rmse_global"].apply(
    lambda d: d.get(variable, np.nan) if isinstance(d, dict) else d
)

scenario_order = list(dict.fromkeys(prediction_df["scenario"].tolist()))
align_values = sorted(prediction_df["lambda_align"].unique())
pred_values = sorted(prediction_df["lambda_pred"].unique(), reverse=True)

# ── Palettes ──────────────────────────────────────────────────────────────
# Sequential (light→dark blue) for the absolute RMSE: pure magnitude, no sign.
sequential_cmap = LinearSegmentedColormap.from_list(
    "sequential_blue", ["#cde2fb", "#6da7ec", "#256abf", "#0d366b"], N=256
)
sequential_cmap.set_bad("#d9d9d6")

# Diverging for the RMSE ratio: red = ratio > 1 (larger error than historical, bad),
# blue = ratio < 1 (smaller error, good), gray = ratio == 1 (no change).
diverging_cmap_rmse = LinearSegmentedColormap.from_list(
    "diverging_blue_red_rmse", ["#2a78d6", "#f0efec", "#e34948"], N=256
)
diverging_cmap_rmse.set_bad("#d9d9d6")

def _make_pivot(values_df, value_col):
    pivot = values_df.pivot_table(index="lambda_pred", columns="lambda_align", values=value_col, aggfunc="first")
    pivot = pivot.reindex(index=pred_values, columns=align_values)
    return pivot

def _plot_heatmap(ax, pivot, title, cmap, vmin=None, vmax=None, norm=None, fmt="{:.2f}"):
    im = ax.imshow(pivot.values, cmap=cmap, vmin=vmin, vmax=vmax, norm=norm, aspect="auto")
    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels([str(c) for c in pivot.columns], rotation=45, ha="right")
    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels([str(i) for i in pivot.index])
    ax.set_xlabel(r"$\lambda_{align}$")
    ax.set_ylabel(r"$\lambda_{pred}$")
    ax.set_title(title, fontsize=10)

    for i in range(pivot.shape[0]):
        for j in range(pivot.shape[1]):
            val = pivot.values[i, j]
            if not np.isnan(val):
                ax.text(j, i, fmt.format(val), ha="center", va="center", fontsize=8, color="black")
    return im

# ── 1) One RMSE heatmap per climate (scenario) ────────────────────────────────
rmse_vmin = 0.0
rmse_vmax = np.nanmax(prediction_df["rmse_value"])

n_scenarios = len(scenario_order)
fig, axes = plt.subplots(1, n_scenarios, figsize=(4 * n_scenarios, 4.2), constrained_layout=True)
if n_scenarios == 1:
    axes = [axes]

im = None
for ax, scenario in zip(axes, scenario_order):
    scenario_df = prediction_df[prediction_df["scenario"] == scenario]
    pivot = _make_pivot(scenario_df, "rmse_value")
    im = _plot_heatmap(ax, pivot, f"RMSE – {scenario}", sequential_cmap, vmin=rmse_vmin, vmax=rmse_vmax)

fig.colorbar(im, ax=axes, shrink=0.8, label=f"Global RMSE for {variable}")
fig.suptitle(f"Global RMSE by scenario ($\\lambda_{{align}}$ vs $\\lambda_{{pred}}$) – {variable}", fontsize=12)
plt.show()

# ── 2) One RMSE ratio heatmap (SSP / historical) per SSP scenario ───────────
_ref_col = next((c for c in ("historical", "hist") if c in scenario_order), None)
if _ref_col is None:
    raise ValueError(f"Historical scenario not found. Available scenarios: {scenario_order}")

_ssp_scenarios = [s for s in scenario_order if s != _ref_col]
if not _ssp_scenarios:
    raise ValueError("No SSP scenario found in the data.")

_ref_df = prediction_df[prediction_df["scenario"] == _ref_col][["lambda_align", "lambda_pred", "rmse_value"]]
_ref_df = _ref_df.rename(columns={"rmse_value": "rmse_ref"})

# A single rmse_value per (lambda_align, lambda_pred, scenario) is guaranteed by summarize_setup_scores,
# so this merge stays bounded (no risk of combinatorial explosion).
_ratio_long = prediction_df[prediction_df["scenario"].isin(_ssp_scenarios)].merge(
    _ref_df, on=["lambda_align", "lambda_pred"], how="left"
)
_ratio_long["ratio_rmse"] = _ratio_long["rmse_value"] / _ratio_long["rmse_ref"]

_ratio_min = np.nanmin(_ratio_long["ratio_rmse"])
_ratio_max = np.nanmax(_ratio_long["ratio_rmse"])
ratio_norm = TwoSlopeNorm(
    vmin=min(_ratio_min, 0.99),
    vcenter=1.0,
    vmax=max(_ratio_max, 1.01),
)

n_ssp = len(_ssp_scenarios)
fig2, axes2 = plt.subplots(1, n_ssp, figsize=(4 * n_ssp, 4.2), constrained_layout=True)
if n_ssp == 1:
    axes2 = [axes2]

im2 = None
for ax, scenario in zip(axes2, _ssp_scenarios):
    scenario_df = _ratio_long[_ratio_long["scenario"] == scenario]
    pivot = _make_pivot(scenario_df, "ratio_rmse")
    im2 = _plot_heatmap(ax, pivot, f"RMSE ratio – {scenario}", diverging_cmap_rmse, norm=ratio_norm)

fig2.colorbar(im2, ax=axes2, shrink=0.8, label=f"RMSE ratio (SSP / {_ref_col}) for {variable}")
fig2.suptitle(f"RMSE ratio vs. {_ref_col} ($\\lambda_{{align}}$ vs $\\lambda_{{pred}}$) – {variable}", fontsize=12)
plt.show()


In [ ]:
from matplotlib.colors import LinearSegmentedColormap, TwoSlopeNorm

# ── Source: the small summary computed setup by setup (Part 1) ───────────────
prediction_df = complete_summary_df[complete_summary_df["component"] == "prediction"].copy()
if prediction_df.empty:
    raise ValueError("No prediction rows found in complete_summary_df.")

prediction_df["rmse_value"] = prediction_df["rmse_global"].apply(
    lambda d: d.get(variable, np.nan) if isinstance(d, dict) else d
)

# ── Restriction to the half-square (0.1,0.1) → (0.85,0.1) → (0.1,0.85) ────────────
_TRI_ALIGN_MIN, _TRI_PRED_MIN = 0.1, 0.1
_TRI_HYPOTENUSE = 0.95  # = 0.85 + 0.1, the constant sum along the hypotenuse

def _filter_triangle(df):
    mask = (
        (df["lambda_align"] >= _TRI_ALIGN_MIN - 1e-9) &
        (df["lambda_pred"] >= _TRI_PRED_MIN - 1e-9) &
        (df["lambda_align"] + df["lambda_pred"] <= _TRI_HYPOTENUSE + 1e-9)
    )
    return df[mask]

prediction_df = _filter_triangle(prediction_df)
if prediction_df.empty:
    raise ValueError("No prediction rows inside the (0.1,0.1)-(0.85,0.1)-(0.1,0.85) triangle.")

scenario_order = list(dict.fromkeys(prediction_df["scenario"].tolist()))

# ── Palettes ──────────────────────────────────────────────────────────────
# Sequential (light→dark blue) for the absolute RMSE: pure magnitude, no sign
# (light = heatmap min value, dark = heatmap max value).
sequential_cmap = LinearSegmentedColormap.from_list(
    "sequential_blue", ["#cde2fb", "#6da7ec", "#256abf", "#0d366b"], N=256
)

# Diverging for the RMSE ratio: red = ratio > 1 (larger error than historical, bad),
# blue = ratio < 1 (smaller error, good), gray = ratio == 1 (no change).
diverging_cmap_rmse = LinearSegmentedColormap.from_list(
    "diverging_blue_red_rmse", ["#2a78d6", "#f0efec", "#e34948"], N=256
)

def _plot_continuous_heatmap(ax, fig, values_df, value_col, title, cmap, kind="sequential", fmt="{:.2f}", cbar_label=""):
    x = values_df["lambda_align"].to_numpy()
    y = values_df["lambda_pred"].to_numpy()
    z = values_df[value_col].to_numpy()

    # Scale specific to this heatmap (computed from its own values only).
    vmin = float(np.nanmin(z))
    vmax = float(np.nanmax(z))

    if kind == "diverging_center1":
        norm = TwoSlopeNorm(vmin=min(vmin, 0.99), vcenter=1.0, vmax=max(vmax, 1.01))
        tcf = ax.tricontourf(x, y, z, levels=21, cmap=cmap, norm=norm)
    else:
        if vmin == vmax:
            vmin, vmax = vmin - 1e-6, vmax + 1e-6
        levels = np.linspace(vmin, vmax, 21)
        tcf = ax.tricontourf(x, y, z, levels=levels, cmap=cmap, vmin=vmin, vmax=vmax)

    ax.scatter(x, y, s=16, facecolor="white", edgecolor="black", linewidth=0.6, zorder=3)
    for xi, yi, zi in zip(x, y, z):
        if np.isfinite(zi):
            ax.annotate(fmt.format(zi), (xi, yi), textcoords="offset points", xytext=(4, 4), fontsize=7)

    ax.set_xlabel(r"$\lambda_{align}$")
    ax.set_ylabel(r"$\lambda_{pred}$")
    ax.set_title(title, fontsize=10)
    ax.set_xlim(_TRI_ALIGN_MIN, 0.85)
    ax.set_ylim(_TRI_PRED_MIN, 0.85)
    ax.set_aspect("equal", adjustable="box")

    fig.colorbar(tcf, ax=ax, shrink=0.85, label=cbar_label)
    return tcf

# ── 1) One continuous RMSE heatmap per climate (scenario), own scale for each ─
n_scenarios = len(scenario_order)
fig, axes = plt.subplots(1, n_scenarios, figsize=(4.6 * n_scenarios, 4.2), constrained_layout=True)
if n_scenarios == 1:
    axes = [axes]

for ax, scenario in zip(axes, scenario_order):
    scenario_df = prediction_df[prediction_df["scenario"] == scenario]
    _plot_continuous_heatmap(
        ax, fig, scenario_df, "rmse_value", f"RMSE – {scenario}", sequential_cmap,
        kind="sequential", cbar_label=f"RMSE for {variable}",
    )

fig.suptitle(f"Global RMSE by scenario ($\\lambda_{{align}}$ vs $\\lambda_{{pred}}$, half-square) – {variable}", fontsize=12)
plt.show()

# ── 2) One continuous RMSE ratio heatmap (SSP / historical), own scale for each ─
_ref_col = next((c for c in ("historical", "hist") if c in scenario_order), None)
if _ref_col is None:
    raise ValueError(f"Historical scenario not found. Available scenarios: {scenario_order}")

_ssp_scenarios = [s for s in scenario_order if s != _ref_col]
if not _ssp_scenarios:
    raise ValueError("No SSP scenario found in the data.")

_ref_df = prediction_df[prediction_df["scenario"] == _ref_col][["lambda_align", "lambda_pred", "rmse_value"]]
_ref_df = _ref_df.rename(columns={"rmse_value": "rmse_ref"})

_ratio_long = prediction_df[prediction_df["scenario"].isin(_ssp_scenarios)].merge(
    _ref_df, on=["lambda_align", "lambda_pred"], how="left"
)
_ratio_long["ratio_rmse"] = _ratio_long["rmse_value"] / _ratio_long["rmse_ref"]

n_ssp = len(_ssp_scenarios)
fig2, axes2 = plt.subplots(1, n_ssp, figsize=(4.6 * n_ssp, 4.2), constrained_layout=True)
if n_ssp == 1:
    axes2 = [axes2]

for ax, scenario in zip(axes2, _ssp_scenarios):
    scenario_df = _ratio_long[_ratio_long["scenario"] == scenario]
    _plot_continuous_heatmap(
        ax, fig2, scenario_df, "ratio_rmse", f"RMSE ratio – {scenario}", diverging_cmap_rmse,
        kind="diverging_center1", cbar_label=f"RMSE ratio (SSP / {_ref_col})",
    )

fig2.suptitle(f"RMSE ratio vs. {_ref_col} ($\\lambda_{{align}}$ vs $\\lambda_{{pred}}$, half-square) – {variable}", fontsize=12)
plt.show()


In [ ]:
from matplotlib.colors import LinearSegmentedColormap, TwoSlopeNorm

if variable == "pr":
    # ── Source: rmse_extreme is already computed by summarize_setup_scores (Part 1),
    # no need to touch truth_values/pred_values here (they don't even exist anymore
    # in complete_summary_df, which only keeps the aggregated metrics).
    prediction_df = complete_summary_df[complete_summary_df["component"] == "prediction"].copy()
    if prediction_df.empty:
        raise ValueError("No prediction rows found in complete_summary_df.")

    prediction_df = prediction_df.dropna(subset=["rmse_extreme"])
    if prediction_df.empty:
        raise ValueError(f"No data above {EXTREME_PRECIP_THRESHOLD} mm/day.")

    scenario_order = list(dict.fromkeys(prediction_df["scenario"].tolist()))
    align_values = sorted(prediction_df["lambda_align"].unique())
    pred_values = sorted(prediction_df["lambda_pred"].unique(), reverse=True)

    # ── Palettes ──────────────────────────────────────────────────────────
    sequential_cmap = LinearSegmentedColormap.from_list(
        "sequential_blue", ["#cde2fb", "#6da7ec", "#256abf", "#0d366b"], N=256
    )
    sequential_cmap.set_bad("#d9d9d6")

    diverging_cmap_rmse = LinearSegmentedColormap.from_list(
        "diverging_blue_red_rmse", ["#2a78d6", "#f0efec", "#e34948"], N=256
    )
    diverging_cmap_rmse.set_bad("#d9d9d6")

    def _make_pivot(values_df, value_col):
        pivot = values_df.pivot_table(index="lambda_pred", columns="lambda_align", values=value_col, aggfunc="first")
        pivot = pivot.reindex(index=pred_values, columns=align_values)
        return pivot

    def _plot_heatmap(ax, pivot, title, cmap, vmin=None, vmax=None, norm=None, fmt="{:.2f}"):
        im = ax.imshow(pivot.values, cmap=cmap, vmin=vmin, vmax=vmax, norm=norm, aspect="auto")
        ax.set_xticks(range(len(pivot.columns)))
        ax.set_xticklabels([str(c) for c in pivot.columns], rotation=45, ha="right")
        ax.set_yticks(range(len(pivot.index)))
        ax.set_yticklabels([str(i) for i in pivot.index])
        ax.set_xlabel(r"$\lambda_{align}$")
        ax.set_ylabel(r"$\lambda_{pred}$")
        ax.set_title(title, fontsize=10)

        for i in range(pivot.shape[0]):
            for j in range(pivot.shape[1]):
                val = pivot.values[i, j]
                if not np.isnan(val):
                    ax.text(j, i, fmt.format(val), ha="center", va="center", fontsize=8, color="black")
        return im

    # ── 1) One extreme-RMSE heatmap per climate (scenario) ────────────────────
    rmse_vmin = 0.0
    rmse_vmax = np.nanmax(prediction_df["rmse_extreme"])

    n_scenarios = len(scenario_order)
    fig, axes = plt.subplots(1, n_scenarios, figsize=(4 * n_scenarios, 4.2), constrained_layout=True)
    if n_scenarios == 1:
        axes = [axes]

    im = None
    for ax, scenario in zip(axes, scenario_order):
        scenario_df = prediction_df[prediction_df["scenario"] == scenario]
        pivot = _make_pivot(scenario_df, "rmse_extreme")
        im = _plot_heatmap(ax, pivot, f"RMSE extreme – {scenario}", sequential_cmap, vmin=rmse_vmin, vmax=rmse_vmax)

    fig.colorbar(im, ax=axes, shrink=0.8, label=f"RMSE on extreme precipitation (≥ {EXTREME_PRECIP_THRESHOLD:.0f} mm/day)")
    fig.suptitle(f"RMSE on extreme precipitation by scenario ($\\lambda_{{align}}$ vs $\\lambda_{{pred}}$)", fontsize=12)
    plt.show()

    # ── 2) One extreme-RMSE ratio heatmap (SSP / historical) per SSP scenario ─
    _ref_col = next((c for c in ("historical", "hist") if c in scenario_order), None)
    if _ref_col is None:
        raise ValueError(f"Historical scenario not found. Available scenarios: {scenario_order}")

    _ssp_scenarios = [s for s in scenario_order if s != _ref_col]
    if not _ssp_scenarios:
        raise ValueError("No SSP scenario found in the data.")

    _ref_df = prediction_df[prediction_df["scenario"] == _ref_col][["lambda_align", "lambda_pred", "rmse_extreme"]]
    _ref_df = _ref_df.rename(columns={"rmse_extreme": "rmse_ref"})

    _ratio_long = prediction_df[prediction_df["scenario"].isin(_ssp_scenarios)].merge(
        _ref_df, on=["lambda_align", "lambda_pred"], how="left"
    )
    _ratio_long["ratio_rmse"] = _ratio_long["rmse_extreme"] / _ratio_long["rmse_ref"]

    _ratio_min = np.nanmin(_ratio_long["ratio_rmse"])
    _ratio_max = np.nanmax(_ratio_long["ratio_rmse"])
    ratio_norm = TwoSlopeNorm(
        vmin=min(_ratio_min, 0.99),
        vcenter=1.0,
        vmax=max(_ratio_max, 1.01),
    )

    n_ssp = len(_ssp_scenarios)
    fig2, axes2 = plt.subplots(1, n_ssp, figsize=(4 * n_ssp, 4.2), constrained_layout=True)
    if n_ssp == 1:
        axes2 = [axes2]

    im2 = None
    for ax, scenario in zip(axes2, _ssp_scenarios):
        scenario_df = _ratio_long[_ratio_long["scenario"] == scenario]
        pivot = _make_pivot(scenario_df, "ratio_rmse")
        im2 = _plot_heatmap(ax, pivot, f"RMSE extreme ratio – {scenario}", diverging_cmap_rmse, norm=ratio_norm)

    fig2.colorbar(im2, ax=axes2, shrink=0.8, label=f"RMSE ratio on extreme precip. (SSP / {_ref_col})")
    fig2.suptitle(f"RMSE ratio on extreme precipitation vs. {_ref_col} ($\\lambda_{{align}}$ vs $\\lambda_{{pred}}$)", fontsize=12)
    plt.show()


In [ ]:
# ── Source: rmse_extreme is already computed by summarize_setup_scores (Part 1),
# no need to touch truth_values/pred_values here (they don't even exist anymore
# in complete_summary_df, which only keeps the aggregated metrics).
prediction_df = complete_summary_df[complete_summary_df["component"] == "prediction"].copy()
if prediction_df.empty:
    raise ValueError("No prediction rows found in complete_summary_df.")

prediction_df = prediction_df.dropna(subset=["rmse_extreme"])
if prediction_df.empty:
    raise ValueError(f"No data above {EXTREME_PRECIP_THRESHOLD} mm/day.")

# ── Restriction to the half-square (0.1,0.1) → (0.85,0.1) → (0.1,0.85) ────────
_TRI_ALIGN_MIN, _TRI_PRED_MIN = 0.1, 0.1
_TRI_HYPOTENUSE = 0.95  # = 0.85 + 0.1, the constant sum along the hypotenuse

def _filter_triangle(df):
    mask = (
        (df["lambda_align"] >= _TRI_ALIGN_MIN - 1e-9) &
        (df["lambda_pred"] >= _TRI_PRED_MIN - 1e-9) &
        (df["lambda_align"] + df["lambda_pred"] <= _TRI_HYPOTENUSE + 1e-9)
    )
    return df[mask]

prediction_df = _filter_triangle(prediction_df)
if prediction_df.empty:
    raise ValueError("No prediction rows inside the (0.1,0.1)-(0.85,0.1)-(0.1,0.85) triangle.")

scenario_order = list(dict.fromkeys(prediction_df["scenario"].tolist()))

# ── Palettes ──────────────────────────────────────────────────────────
# Sequential (light→dark blue) for the absolute extreme RMSE: pure magnitude,
# no sign (light = heatmap min value, dark = heatmap max value).
sequential_cmap = LinearSegmentedColormap.from_list(
    "sequential_blue", ["#cde2fb", "#6da7ec", "#256abf", "#0d366b"], N=256
)

# Diverging for the extreme RMSE ratio: red = ratio > 1 (larger error than
# historical, bad), blue = ratio < 1 (smaller error, good), white = ratio == 1.
diverging_cmap_rmse = LinearSegmentedColormap.from_list(
    "diverging_blue_red_rmse", ["#2a78d6", "#ffffff", "#e34948"], N=256
)

def _plot_continuous_heatmap(ax, fig, values_df, value_col, title, cmap, kind="sequential", fmt="{:.2f}", cbar_label=""):
    x = values_df["lambda_align"].to_numpy()
    y = values_df["lambda_pred"].to_numpy()
    z = values_df[value_col].to_numpy()

    # Scale specific to this heatmap (computed from its own values only).
    vmin = float(np.nanmin(z))
    vmax = float(np.nanmax(z))

    if kind == "diverging_center1":
        norm = TwoSlopeNorm(vmin=min(vmin, 0.99), vcenter=1.0, vmax=max(vmax, 1.01))
        tcf = ax.tricontourf(x, y, z, levels=21, cmap=cmap, norm=norm)
    else:
        if vmin == vmax:
            vmin, vmax = vmin - 1e-6, vmax + 1e-6
        levels = np.linspace(vmin, vmax, 21)
        tcf = ax.tricontourf(x, y, z, levels=levels, cmap=cmap, vmin=vmin, vmax=vmax)

    ax.scatter(x, y, s=16, facecolor="white", edgecolor="black", linewidth=0.6, zorder=3)
    for xi, yi, zi in zip(x, y, z):
        if np.isfinite(zi):
            ax.annotate(fmt.format(zi), (xi, yi), textcoords="offset points", xytext=(4, 4), fontsize=7)

    ax.set_xlabel(r"$\lambda_{align}$")
    ax.set_ylabel(r"$\lambda_{pred}$")
    ax.set_title(title, fontsize=10)
    ax.set_xlim(_TRI_ALIGN_MIN, 0.85)
    ax.set_ylim(_TRI_PRED_MIN, 0.85)
    ax.set_aspect("equal", adjustable="box")

    fig.colorbar(tcf, ax=ax, shrink=0.85, label=cbar_label)
    return tcf

# ── 1) One continuous extreme-RMSE heatmap per climate, own scale for each ─
n_scenarios = len(scenario_order)
fig, axes = plt.subplots(1, n_scenarios, figsize=(4.6 * n_scenarios, 4.2), constrained_layout=True)
if n_scenarios == 1:
    axes = [axes]

for ax, scenario in zip(axes, scenario_order):
    scenario_df = prediction_df[prediction_df["scenario"] == scenario]
    _plot_continuous_heatmap(
        ax, fig, scenario_df, "rmse_extreme", f"RMSE extreme – {scenario}", sequential_cmap,
        kind="sequential", cbar_label=f"RMSE on extreme precip. (≥ {EXTREME_PRECIP_THRESHOLD:.0f} mm/day)",
    )

fig.suptitle(f"RMSE on extreme precipitation by scenario ($\\lambda_{{align}}$ vs $\\lambda_{{pred}}$, half-square)", fontsize=12)
plt.show()

# ── 2) One continuous extreme-RMSE ratio heatmap (SSP / historical), own scale for each ─
_ref_col = next((c for c in ("historical", "hist") if c in scenario_order), None)
if _ref_col is None:
    raise ValueError(f"Historical scenario not found. Available scenarios: {scenario_order}")

_ssp_scenarios = [s for s in scenario_order if s != _ref_col]
if not _ssp_scenarios:
    raise ValueError("No SSP scenario found in the data.")

_ref_df = prediction_df[prediction_df["scenario"] == _ref_col][["lambda_align", "lambda_pred", "rmse_extreme"]]
_ref_df = _ref_df.rename(columns={"rmse_extreme": "rmse_ref"})

_ratio_long = prediction_df[prediction_df["scenario"].isin(_ssp_scenarios)].merge(
    _ref_df, on=["lambda_align", "lambda_pred"], how="left"
)
_ratio_long["ratio_rmse"] = _ratio_long["rmse_extreme"] / _ratio_long["rmse_ref"]

n_ssp = len(_ssp_scenarios)
fig2, axes2 = plt.subplots(1, n_ssp, figsize=(4.6 * n_ssp, 4.2), constrained_layout=True)
if n_ssp == 1:
    axes2 = [axes2]

for ax, scenario in zip(axes2, _ssp_scenarios):
    scenario_df = _ratio_long[_ratio_long["scenario"] == scenario]
    _plot_continuous_heatmap(
        ax, fig2, scenario_df, "ratio_rmse", f"RMSE extreme ratio – {scenario}", diverging_cmap_rmse,
        kind="diverging_center1", cbar_label=f"RMSE ratio on extreme precip. (SSP / {_ref_col})",
    )

fig2.suptitle(f"RMSE ratio on extreme precipitation vs. {_ref_col} ($\\lambda_{{align}}$ vs $\\lambda_{{pred}}$, half-square)", fontsize=12)
plt.show()


## Part 4 - Plotting history results

In [ ]:
# Charging CERA historical dataframes for each (lambda_align, lambda_pred) pair
cera_history_dfs = {
    (lambda_align, lambda_pred): _load_history_df(_cera_history_path(lambda_align, lambda_pred))
    for lambda_align, lambda_pred in lambda_pairs
}

### *CERA*

In [ ]:
fig, ax = plt.subplots(figsize=(11, 6), constrained_layout=True)

lambda_align = 0.1
lambda_pred = 0.1

history_df = cera_history_dfs[(lambda_align, lambda_pred)]

colors = {
    "recon": "#2C7FB8",
    "align": "#7F7F7F",
    "pred":  "#6A3D9A", 
    "total": "#D62728",  
}

loss_groups = {
    "recon": ["train_recon_loss", "val_recon_loss"],
    "align": ["train_align_loss", "val_align_loss"],
    "pred":  ["train_pred_loss", "val_pred_loss"],
    "total": ["train_total_loss", "val_total_loss"],
}

for group, cols in loss_groups.items():
    for col in cols:
        is_train = "train" in col
        
        ax.plot(
            history_df["epoch"],
            history_df[col],
            linestyle="--" if is_train else "-",   
            linewidth=1,
            color=colors[group],
            alpha=0.9 if not is_train else 0.7,
            label=col.replace("_", " ")
        )

ax.set_yscale("log")
ax.set_title(f"CERA training dynamics (log scale) — $\\lambda_{{align}}$={lambda_align:g}, $\\lambda_{{pred}}$={lambda_pred:g}", fontsize=13)
ax.set_xlabel("Epoch")
ax.set_ylabel("Loss (log scale)")

ax.grid(True, which="both", axis="y", alpha=0.25)
ax.grid(True, which="major", axis="x", alpha=0.15)

ax.legend(title="Loss components", ncol=2, fontsize=9)

plt.show()